# Imputation of Missing Weekly Observations

Before fitting forecasting models, missing values must be handled carefully to avoid bias and information leakage.

In this notebook, we implement imputation strategies that:
- Use **only training data**
- Respect the temporal ordering of observations
- Adapt to the size and structure of missing data gaps

The imputed series are then used as inputs for baseline and classical forecasting models.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent   # go up one level
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
pip install -U scikit-learn

Note: you may need to restart the kernel to use updated packages.


## Imputation strategy

Two types of missing data situations are considered:

- **Isolated missing observations**, which are interpolated as the mean of their 2 neighboring values
- **Grouped missing observations**, which are imputed using k-nearest neighbors on local temporal windows

Importantly, imputation is applied **only to the training period**, ensuring that no future information is used when predicting unseen data.


In [4]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer

def find_nan_gaps(s: pd.Series):
    """
    s: Serie indexada por fechas, con NaN.
    Devuelve lista de gaps: [(start_ts, end_ts, length), ...] con extremos inclusive.
    """
    is_na = s.isna()
    if is_na.sum() == 0:
        return []

    block_id = (is_na != is_na.shift(fill_value=False)).cumsum()
    gaps = []
    for _, block in is_na.groupby(block_id):
        if block.iloc[0]:  # gap
            start = block.index[0]
            end = block.index[-1]
            length = int(block.sum())
            gaps.append((start, end, length))
    return gaps


## **2. Imputation of isolated one-week gaps**

Single missing observations are imputed using the mean of their two neighbouring values.

For a missing value at week (t), the imputed value is computed as the average of the previous and next observed weeks.

This rule is only applied when both neighbouring values are available. If one of the two neighbours is also missing, the value is not imputed.

This provides a simple and conservative way to fill isolated gaps without using information from distant observations.


In [5]:
def impute_singletons_neighbor_mean(s: pd.Series) -> pd.Series:
    """
    Imputa solo gaps de longitud 1 usando (t-1 + t+1)/2.
    """
    s = s.copy()
    gaps = find_nan_gaps(s)

    for start, end, length in gaps:
        if length == 1:
            t = start
            # vecinos
            prev_t = s.index[s.index.get_loc(t) - 1] if s.index.get_loc(t) - 1 >= 0 else None
            next_t = s.index[s.index.get_loc(t) + 1] if s.index.get_loc(t) + 1 < len(s.index) else None

            if prev_t is not None and next_t is not None:
                if pd.notna(s.loc[prev_t]) and pd.notna(s.loc[next_t]):
                    s.loc[t] = 0.5 * (s.loc[prev_t] + s.loc[next_t])
    return s


## **3. KNN imputation of moderate grouped gaps**

Grouped gaps with length between 2 and `max_gap_knn` are imputed using a local K-nearest neighbours approach.

For each missing block, a local temporal window is built around the gap. The window size is set as twice the gap length on each side.

This creates a local block containing the missing values and the nearby observations before and after the gap.

Within this block, lag and lead variables are created from the original series. These variables are used as features for the `KNNImputer`. After the imputation, only the missing values inside the original gap are replaced.

This method allows moderate gaps to be filled using local temporal patterns, while avoiding the imputation of long missing periods where the available information may not be reliable.


In [6]:
def build_lag_features(series: pd.Series, n_lags: int = 8):
    df = pd.DataFrame({"value": series})
    for k in range(1, n_lags + 1):
        df[f"lag_{k}"] = series.shift(k)
        df[f"lead_{k}"] = series.shift(-k)
    return df

def impute_gaps_knn(s: pd.Series, max_gap_knn: int = 8, n_lags: int = 8, k_neighbors: int = 5) -> pd.Series:
    """
    imputes gaps of length 2..max_gap_knn using KNN inside local windows
    """
    s = s.copy()
    gaps = find_nan_gaps(s)

    for start, end, length in gaps:
        if length < 2 or length > max_gap_knn:
            continue

        # window = twize the gap
        w = 2 * length

        # locate positions
        idx = s.index
        start_i = idx.get_loc(start)
        end_i = idx.get_loc(end)

        left_i = max(0, start_i - w)
        right_i = min(len(idx) - 1, end_i + w)

        block_idx = idx[left_i:right_i+1]
        block = s.loc[block_idx]

        # if block has too many gaps (few info), jump
        if block.notna().sum() < max(5, 2*length):
            continue

        X = build_lag_features(block, n_lags=n_lags)

        imputer = KNNImputer(n_neighbors=k_neighbors, weights="distance")
        X_imp = pd.DataFrame(imputer.fit_transform(X), index=X.index, columns=X.columns)

        # we update only the part that was a gap
        s.loc[start:end] = X_imp.loc[start:end, "value"]

    return s


## **4. Final series-level imputation function**

The final imputation function applies the complete procedure to each individual time series.

First, the series is sorted by date, duplicated dates are removed, and the data are reindexed to a complete weekly calendar. This ensures that missing weeks are explicitly represented as missing values.

The option `trim_to_first_obs=True` removes the empty initial part of the series before the first observed value. This avoids treating long initial periods with no data as internal gaps.

After this preprocessing step, the imputation rules are applied in sequence:

1. Isolated one-week gaps are filled using the mean of the two neighbouring observations.
2. Moderate grouped gaps are imputed using the KNN-based procedure.
3. Gaps longer than `max_gap_knn` are not imputed and remain as missing values.

This keeps the imputation conservative and avoids filling long periods with insufficient local information.


In [7]:
def impute_series_weekly(df_loc: pd.DataFrame,
                         calendar: pd.DatetimeIndex,
                         trim_to_first_obs: bool = True,
                         max_gap_knn: int = 8,
                         n_lags: int = 8,
                         k_neighbors: int = 5):
    """
    df_loc: dataframe from a country with columns [truth_date, value]
    calendar: full weekly index
    """
    g = df_loc.copy()
    g["truth_date"] = pd.to_datetime(g["truth_date"])
    g = g.sort_values("truth_date").drop_duplicates("truth_date", keep="last").set_index("truth_date")
    s = g["value"].reindex(calendar)

    if trim_to_first_obs:
        first = s.first_valid_index()
        if first is not None:
            s = s.loc[first:]  # trims initial NANs

    # 1) singleton gaps
    s = impute_singletons_neighbor_mean(s)

    # 2) grouped gaps (moderate) with KNN
    s = impute_gaps_knn(s, max_gap_knn=max_gap_knn, n_lags=n_lags, k_neighbors=k_neighbors)

    return s
